In [1]:
from smolagents import CodeAgent, TransformersModel, tool
from transformers import AutoTokenizer, pipeline

In [2]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime

%matplotlib inline
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl.meta.data_processor import DataProcessor

from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline
from pprint import pprint

import sys
sys.path.append("../FinRL")

import itertools

C:\Users\silve\anaconda3\envs\CM3070-FP\lib\site-packages\pyfolio\pos.py:25: UserWarning: Module "zipline.assets" not found; multipliers will not be applied to position notionals.
  warnings.warn(


In [3]:
from finrl import config
from finrl import config_tickers
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)
check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])

In [1]:
import reservoirpy as rpy
from reservoirpy.nodes import Reservoir, Ridge
from reservoirpy.observables import mse, nrmse, rmse
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
model_id = "Qwen/Qwen3-1.7B"

model = TransformersModel(model_id=model_id, max_new_tokens=300)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
ticker_list = ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SWH', 'TRV', 'UNH', 'V', 'VZ', 'WMT']

['AXP',
 'AMGN',
 'AAPL',
 'BA',
 'CAT',
 'CSCO',
 'CVX',
 'GS',
 'HD',
 'HON',
 'IBM',
 'INTC',
 'JNJ',
 'KO',
 'JPM',
 'MCD',
 'MMM',
 'MRK',
 'MSFT',
 'NKE',
 'PG',
 'TRV',
 'UNH',
 'CRM',
 'VZ',
 'V',
 'WBA',
 'WMT',
 'DIS',
 'DOW']

In [5]:
def load_or_update_data(verbose=True):
    today_str = datetime.datetime.today().strftime("%Y-%m-%d")
    data_dir = "./data"
    
    # Create data folder if it doesn't exist
    os.makedirs(data_dir, exist_ok=True)
    
    # Look for file ending with the given string
    matching_files = [
        f for f in os.listdir(data_dir)
        if f.endswith("dow30.csv")
    ]
    
    if matching_files:
        file = matching_files[0]
        if file.startswith(today_str):
            if verbose:
                print(f"✅ Found up-to-date file: {file}")
            return pd.read_csv(os.path.join(data_dir, file))
        else:
            if verbose:
                print(f"🗑 Deleting stale file: {file}")
            os.remove(os.path.join(data_dir, file))

    # Fetch new data
    if verbose:
        print("⬇ Fetching fresh data...")
    df = YahooDownloader(
        start_date=TRAIN_START_DATE,
        end_date=TRADE_END_DATE,
        ticker_list = ['AAPL', 
                       'AMGN',
                       'AMZN',
                       'AXP',
                       'BA',
                       'CAT',
                       'CRM',
                       'CSCO',
                       'CVX',
                       'DIS',
                       'GS',
                       'HD',
                       'HON',
                       'IBM',
                       'JNJ',
                       'JPM',
                       'KO',
                       'MCD',
                       'MMM',
                       'MRK',
                       'MSFT',
                       'NKE',
                       'NVDA',
                       'PG',
                       'SWH',
                       'TRV',
                       'UNH',
                       'V',
                       'VZ',
                       'WMT']
    ).fetch_data()

    fe = FeatureEngineer(
                    use_technical_indicator=True,
                    tech_indicator_list = INDICATORS,
                    use_vix=True,
                    use_turbulence=True,
                    user_defined_feature = False)

    processed = fe.preprocess_data(df)

    list_ticker = processed["tic"].unique().tolist()
    list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
    combination = list(itertools.product(list_date,list_ticker))

    processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
    processed_full = processed_full[processed_full['date'].isin(processed['date'])]
    processed_full = processed_full.sort_values(['date','tic'])

    processed_full = processed_full.fillna(0)

    

    filename = f"{today_str}_dow30.csv"
    df.to_csv(os.path.join(data_dir, filename), index=False)
    if verbose:
        print(f"💾 Saved new file: {filename}")
    return df

In [7]:
@tool
def fetch_latest_data(ticker: str) -> str:

    """
    Fetch the latest available stock data for a specific ticker.

    Args:
        ticker (str): The stock ticker symbol.

    Returns:
        str: A full-sentence summary of the stock's latest performance, like:
        "On 2025-06-06, AAPL opened at $203.00, reached a high of $205.70, a low of $202.05, and closed at $203.92 with a volume of 46,539,200."
    """
    
    from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
    
    df = load_or_update_data(verbose=False)
    data = df[df['tic'] == ticker].tail(1)

    if data.empty:
        return f"No data found for {ticker}."

    data = data.iloc[0]

    summary = (
        f"On {data['date']}, {ticker.upper()} opened at ${data['open']:.2f}, "
        f"reached a high of ${data['high']:.2f}, a low of ${data['low']:.2f}, "
        f"and closed at ${data['close']:.2f} with a volume of {int(data['volume']):,}."
        f"would you like to have next day close prediction or trading strategies?"
    )

    return summary

In [8]:
agent = CodeAgent(tools=[fetch_latest_data], model=model,
                  additional_authorized_imports=['pandas', 'datetime', 'reservoirpy'],
                  add_base_tools=True)

#agent.run("Could you give me latest data for APPLE?",)

In [ ]:
messages = []

# Initial prompt
initial_prompt = {
    "role": "system",
    "content": [
        {
            "type": "text",
            "text": "You are Qwen, a helpful and funny assistant. Always respond helpfully and occasionally make witty jokes related to finance, but don't repeat your introduction."
        }
    ]
}

intro_prompt = {
    "role": "system",
    "content": [{"type": "text", "text": "Offer stock investing advice, ask me which stock I am interested in. Mention that it is currently limited to DOW30"}]
}
response = model([initial_prompt, intro_prompt], max_new_tokens=500)
assistant_message = response.content
print("Assistant:", assistant_message)

# Save to history
messages.append(intro_prompt)
messages.append({
    "role": "assistant",
    "content": [{"type": "text", "text": assistant_message}]
})

# Start chat loop
while True:
    user_input = input("Enter a message (type 'exit' to quit): ")
    if user_input.lower() == "exit":
        print("Exiting chat...")
        break

    messages.append({
        "role": "user",
        "content": [{"type": "text", "text": user_input}]
    })

    # If user asks for stock data, use agent instead of model
    if "latest data" in user_input.lower() or "stock" in user_input.lower():
        print("🔍 Agent is handling your request...")
        try:
            result = agent.run(user_input)
            assistant_message = result if isinstance(result, str) else str(result)
        except Exception as e:
            assistant_message = f"❌ Error from agent: {str(e)}"
    else:
        # Otherwise, use normal chat model
        response = model(messages, max_new_tokens=500)
        assistant_message = response.content

    print("Assistant:", assistant_message)

    messages.append({
        "role": "assistant",
        "content": [{"type": "text", "text": assistant_message}]
    })

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Assistant: </think>

Hey there! I'm Qwen, your friendly and witty stock investing buddy. I can help you with DOW30 insights, but I need to know which stock you're interested in to give you the best advice. Let's keep things light and fun—maybe I can even throw in a joke about market trends! 📈💸


Enter a message (type 'exit' to quit):  could you give me the latest data for Apple and American Express


🔍 Agent is handling your request...


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ could you give me the latest data for Apple and American Express                                                │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3-1.7B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  apple_data = fetch_latest_data("AAPL")                                                                           
  american_express_data = fetch_latest_data("WBA")                                                                 
  final_answer(f"Apple: {apple_data}, American Express: {american_express_data}")                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: Apple: On 2021-11-30, AAPL opened at $157.03, reached a high of $162.46, a low of $156.96, and 
closed at $162.24 with a volume of 174,048,100.would you like to have next day close prediction or trading 
strategies?, American Express: On 2021-11-30, WBA opened at $37.96, reached a high of $38.33, a low of $36.60, and 
closed at $36.93 with a volume of 13,238,100.would you like to have next day close prediction or trading 
strategies?

[Step 1: Duration 203.00 seconds| Input tokens: 2,254 | Output tokens: 76]

Assistant: Apple: On 2021-11-30, AAPL opened at $157.03, reached a high of $162.46, a low of $156.96, and closed at $162.24 with a volume of 174,048,100.would you like to have next day close prediction or trading strategies?, American Express: On 2021-11-30, WBA opened at $37.96, reached a high of $38.33, a low of $36.60, and closed at $36.93 with a volume of 13,238,100.would you like to have next day close prediction or trading strategies?
